In [12]:
import pandas as pd
import gurobipy as gp
import folium

# Data
data = pd.read_csv('/Users/linus/PycharmProjects/JupyterProject1/data/cn.csv')
data = data[(data['capital'] == 'admin') | (data['capital'] == 'primary')]

n = data.shape[0]
cities = data['city'].values
loc_x_vals = data['lat'].values
loc_y_vals = data['lng'].values
loc_x = dict(zip(cities, loc_x_vals))
loc_y = dict(zip(cities, loc_y_vals))

I = cities
J = cities
p = round(n * .2)

c={(i,j):((loc_x[i]-loc_x[j])**2+(loc_y[i]-loc_y[j])**2)**.5 for i in I for j in J}

# Model
m = gp.Model('p-median')

# Variables
x = m.addVars(I, vtype=gp.GRB.BINARY, name='x')
y = m.addVars(I, J, vtype=gp.GRB.BINARY, name='y')

# Objective
m.setObjective(y.prod(c), gp.GRB.MINIMIZE)

# Constraints
m.addConstrs((y.sum(i, '*') == 1 for i in I), name='cover')
m.addConstr(x.sum() == p, name='p_location')
m.addConstrs((y[i, j] <= x[j] for i in I for j in I), name='capacity')

# Solve
m.params.timeLimit = 600
m.optimize()

# print
print('the solution time is', m.Runtime)
print('the MIP gap is', m.MIPGap)

# ---analysis---
if m.status != gp.GRB.INFEASIBLE:
    pmedian_map = folium.Map(location=[34.32,108.55], zoom_start=4)

    # mark the capitals
    cities_selected = [j for j in J if x[j].x > 0.9]
    for i in I:
        if i in cities_selected:
            folium.Marker(location=[loc_x[i], loc_y[i]], popup=i, icon=folium.Icon(color='red')).add_to(pmedian_map)
        else:
            folium.Marker(location=[loc_x[i], loc_y[i]], popup=i).add_to(pmedian_map)

    # plot assignment decisions
    assignments = [(i, j) for i in I for j in J if y[i, j].x > 0.9]

    # plot results
    for i, j in assignments:
        folium.PolyLine(locations=[(loc_x[i], loc_y[i]), (loc_x[j], loc_y[j])]).add_to(pmedian_map)

pmedian_map


Set parameter TimeLimit to value 600
Gurobi Optimizer version 12.0.1 build v12.0.1rc0 (mac64[arm] - Darwin 24.4.0 24E263)

CPU model: Apple M3 Pro
Thread count: 11 physical cores, 11 logical processors, using up to 11 threads

Non-default parameters:
TimeLimit  600

Optimize a model with 1057 rows, 1056 columns and 3104 nonzeros
Model fingerprint: 0x30d1817b
Variable types: 0 continuous, 1056 integer (1056 binary)
Coefficient statistics:
  Matrix range     [1e+00, 1e+00]
  Objective range  [1e+00, 4e+01]
  Bounds range     [1e+00, 1e+00]
  RHS range        [1e+00, 6e+00]
Presolve time: 0.00s
Presolved: 1057 rows, 1056 columns, 3104 nonzeros
Variable types: 0 continuous, 1056 integer (1056 binary)
Found heuristic solution: objective 160.7314345

Root relaxation: objective 1.127420e+02, 255 iterations, 0.00 seconds (0.00 work units)

    Nodes    |    Current Node    |     Objective Bounds      |     Work
 Expl Unexpl |  Obj  Depth IntInf | Incumbent    BestBd   Gap | It/Node Time

     